Data Loading and Preparation
Workflow: Load the airbnb.xlsx - AirBnB_NYC.csv dataset. Convert the Host Since column to a proper datetime format. Analytical Question: In a markdown cell, explain what kind of new, valuable feature you could create from the Host Since column. How might this new feature help the model predict prices? Screenshot Requirement: Upload a screenshot showing your code, the .info() output confirming the new data type, and your markdown answer.

# Data Loading and Preparation Workflow:

In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("AirBnB_NYC.csv")   # or pd.read_excel("airbnb.xlsx") if using Excel

# Convert Host Since to datetime
df["Host Since"] = pd.to_datetime(df["Host Since"])

# Check datatypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30475 entries, 0 to 30474
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Host Id                     30475 non-null  int64         
 1   Host Since                  30475 non-null  datetime64[ns]
 2   Name                        30475 non-null  object        
 3   Neighbourhood               30475 non-null  object        
 4   Property Type               30472 non-null  object        
 5   Review Scores Rating (bin)  22155 non-null  float64       
 6   Room Type                   30475 non-null  object        
 7   Zipcode                     30341 non-null  float64       
 8   Beds                        30390 non-null  float64       
 9   Number of Records           30475 non-null  int64         
 10  Number Of Reviews           30475 non-null  int64         
 11  Price                       30475 non-null  object    

### New Feature from `Host Since`

A valuable new feature that can be created from the `Host Since` column is **host tenure**, which represents how long a host has been active on Airbnb.

For example, I could calculate the number of years or days between the host's start date and today. This feature may help the model predict prices because hosts with more experience may understand the market better, set more competitive prices, and provide better service. More experienced hosts may also have stronger reputations and better reviews, which can influence listing prices.

Workflow: The columns related to review scores have missing values. Decide on an appropriate imputation strategy to handle them. Analytical Question: In a markdown cell, justify your choice of imputation strategy for the Review Scores Rating column. Did you use the mean, median, or another value? Why was your choice appropriate for this column? Screenshot Requirement: Upload a screenshot showing your imputation code, the output of .isnull().sum() after imputation, and your markdown justification.

# Handling Missing Data

In [2]:
# Impute missing values with median
review_median = df["Review Scores Rating"].median()
review_bin_median = df["Review Scores Rating (bin)"].median()

df["Review Scores Rating"] = df["Review Scores Rating"].fillna(review_median)
df["Review Scores Rating (bin)"] = df["Review Scores Rating (bin)"].fillna(review_bin_median)

# Check missing values after imputation
df[["Review Scores Rating", "Review Scores Rating (bin)"]].isnull().sum()

Review Scores Rating          0
Review Scores Rating (bin)    0
dtype: int64

### Handling Missing Data in Review Scores Rating

I used the **median** to fill missing values in the `Review Scores Rating` column.

The median was appropriate because this column contains numerical rating values, and ratings are often not perfectly normally distributed. Since some listings may have unusually low or high ratings, the median is a better measure of central tendency than the mean because it is less affected by outliers.

Using the median helps preserve the typical review score in the dataset while avoiding distortion from extreme values. This makes it a reliable imputation strategy for improving data quality before modeling.

Advanced Feature Engineering
Workflow: Engineer a new interaction feature by combining Neighbourhood and Room Type. Prepare two versions of your dataset for modeling: one without your new feature, and one with it. Train and evaluate a Random Forest Regressor on both datasets and compare their RMSE scores. Analytical Question: In a markdown cell, present the RMSE scores from both models. Did your new feature improve the model's performance? Explain why an interaction feature like this might capture more nuanced information. Screenshot Requirement: Upload a screenshot showing the code where you create the new feature, the two RMSE scores from your model evaluations, and your markdown answer explaining the impact.

# Advanced Feature Engineering

In [3]:
# Remove leading/trailing spaces from all column names
df.columns = df.columns.str.strip()

# Check cleaned column names
print(df.columns.tolist())

['Host Id', 'Host Since', 'Name', 'Neighbourhood', 'Property Type', 'Review Scores Rating (bin)', 'Room Type', 'Zipcode', 'Beds', 'Number of Records', 'Number Of Reviews', 'Price', 'Review Scores Rating']


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Clean column names
df.columns = df.columns.str.strip()

# Make a copy
df_model = df.copy()

# Convert Price to numeric
df_model["Price"] = pd.to_numeric(df_model["Price"], errors="coerce")

# Create interaction feature
df_model["Neighbourhood_RoomType"] = (
    df_model["Neighbourhood"].astype(str) + "_" + df_model["Room Type"].astype(str)
)

# Drop rows where target is missing
df_model = df_model.dropna(subset=["Price"])

# Convert Host Since into numeric form
df_model["Host Since"] = pd.to_datetime(df_model["Host Since"], errors="coerce")
df_model["Host Since"] = df_model["Host Since"].map(pd.Timestamp.toordinal)

# Features
features_without = [
    "Host Id", "Host Since", "Neighbourhood", "Property Type",
    "Review Scores Rating (bin)", "Room Type", "Zipcode",
    "Beds", "Number of Records", "Number Of Reviews",
    "Review Scores Rating"
]

features_with = features_without + ["Neighbourhood_RoomType"]

target = "Price"

X_without = df_model[features_without]
X_with = df_model[features_with]
y = df_model[target]

# Train/test split
X_train_wo, X_test_wo, y_train, y_test = train_test_split(
    X_without, y, test_size=0.2, random_state=42
)

X_train_w, X_test_w, _, _ = train_test_split(
    X_with, y, test_size=0.2, random_state=42
)

# Identify numeric/categorical columns
cat_cols_wo = X_train_wo.select_dtypes(include=["object"]).columns.tolist()
num_cols_wo = X_train_wo.select_dtypes(exclude=["object"]).columns.tolist()

cat_cols_w = X_train_w.select_dtypes(include=["object"]).columns.tolist()
num_cols_w = X_train_w.select_dtypes(exclude=["object"]).columns.tolist()

# Preprocessing
preprocessor_wo = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols_wo),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols_wo)
])

preprocessor_w = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols_w),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols_w)
])

# Models
model_wo = Pipeline([
    ("preprocessor", preprocessor_wo),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

model_w = Pipeline([
    ("preprocessor", preprocessor_w),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train models
model_wo.fit(X_train_wo, y_train)
model_w.fit(X_train_w, y_train)

# Predictions
pred_wo = model_wo.predict(X_test_wo)
pred_w = model_w.predict(X_test_w)

# RMSE
rmse_wo = np.sqrt(mean_squared_error(y_test, pred_wo))
rmse_w = np.sqrt(mean_squared_error(y_test, pred_w))

print("RMSE without interaction feature:", rmse_wo)
print("RMSE with interaction feature:", rmse_w)

RMSE without interaction feature: 79.02728995150133
RMSE with interaction feature: 79.16697223608823


### RMSE Comparison

I trained two Random Forest Regressor models: one without the interaction feature and one with the new `Neighbourhood_RoomType` feature.

- **RMSE without interaction feature:** 79.03
- **RMSE with interaction feature:** 79.17

The interaction feature did **not** improve the model's performance, because the RMSE became slightly higher after adding it. Since lower RMSE indicates better predictions, the model without the new feature performed slightly better.

A feature like `Neighbourhood_RoomType` can still be useful because it captures more specific combinations of location and room type. For example, an entire home in Manhattan may follow a different pricing pattern than a private room in Brooklyn. This kind of interaction can help a model learn more nuanced relationships between features.

In this case, however, the Random Forest model may have already captured much of that relationship from the original `Neighbourhood` and `Room Type` columns separately, so the added interaction feature did not provide enough new information to improve prediction accuracy.

Final Model Training and Evaluation
Workflow: Using your best-performing dataset from the previous task, prepare the data, split it, train your final Random Forest Regressor, and evaluate its Root Mean Squared Error (RMSE). Analytical Question: In a markdown cell, explain what your final model's RMSE value means in practical terms. For example, if your RMSE is 50, what does that tell a host about the accuracy of the predicted price? Screenshot Requirement: Upload a screenshot showing the code for training and evaluation, the printed RMSE score, and your markdown interpretation.

# Final Model Training and Evaluation

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Clean column names
df.columns = df.columns.str.strip()

# Make a copy
df_final = df.copy()

# Convert columns
df_final["Price"] = pd.to_numeric(df_final["Price"], errors="coerce")
df_final["Host Since"] = pd.to_datetime(df_final["Host Since"], errors="coerce")
df_final["Host Since"] = df_final["Host Since"].map(pd.Timestamp.toordinal)

# Drop rows with missing target
df_final = df_final.dropna(subset=["Price"])

# Best-performing feature set from Step 3: WITHOUT interaction feature
features_final = [
    "Host Id", "Host Since", "Neighbourhood", "Property Type",
    "Review Scores Rating (bin)", "Room Type", "Zipcode",
    "Beds", "Number of Records", "Number Of Reviews",
    "Review Scores Rating"
]

target = "Price"

X = df_final[features_final]
y = df_final[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Separate numeric and categorical columns
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# Final model
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train model
final_model.fit(X_train, y_train)

# Predict
y_pred = final_model.predict(X_test)

# Evaluate RMSE
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Final Model RMSE:", final_rmse)

Final Model RMSE: 79.02728995150133


### Final Model RMSE Interpretation

The final Random Forest Regressor model achieved an **RMSE of 79.03**.

In practical terms, RMSE tells us the typical size of the model's prediction error in the same unit as the target variable, which is price. This means that the predicted Airbnb price is usually off by about **$79.03** on average.

For a host, this means the model's estimated listing price will often be close to the actual price, but it may still be around
dollar $79 higher or lower than the true value. So if the model predicts a

price of dollar $200, the actual price might reasonably be somewhere around

$121 to

$279, on average.

A lower RMSE would indicate more accurate predictions, while a higher RMSE would mean the model's predictions are less precise. Overall, this RMSE gives a practical estimate of the model's average pricing error.

## Save the Trained Model

In [8]:
import pickle

with open("model.pkl", "wb") as file:
    pickle.dump(final_model, file)

print("Model saved successfully.")

Model saved successfully.


### Why a Price Range Is More Useful Than a Single Predicted Price

Presenting a price range is more useful for a host because it reflects the uncertainty in the model’s prediction. A single predicted price may give the impression that the estimate is exact, but in reality, machine learning predictions usually have some error.

A price range gives the host more flexibility and helps them make a better decision about how to price their listing. For example, if the predicted range is $120 to $200, the host can choose a price within that range depending on demand, season, location, and amenities.